# Federated Learning Security and Byzantine Attack

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from scipy.spatial.distance import cosine
from scipy.stats import entropy
import hashlib
from collections import defaultdict, deque

class FederatedLearningSecurityFramework:
    def __init__(self, num_clients=10, byzantine_ratio=0.3):
        self.num_clients = num_clients
        self.byzantine_ratio = byzantine_ratio
        self.global_model = LogisticRegression(max_iter=1000, random_state=42)
        self.client_models = [LogisticRegression(max_iter=500, random_state=i) for i in range(num_clients)]
        self.scaler = StandardScaler()
        self.aggregation_history = deque(maxlen=50)
        self.trust_scores = np.ones(num_clients)
        self.attack_detection_threshold = 0.7

    def simulate_byzantine_attacks(self, local_updates, attack_type='model_poisoning'):
        byzantine_clients = np.random.choice(self.num_clients,
                                           int(self.num_clients * self.byzantine_ratio),
                                           replace=False)

        poisoned_updates = local_updates.copy()
        for client_id in byzantine_clients:
            if attack_type == 'model_poisoning':
                poisoned_updates[client_id] = self.model_poisoning_attack(local_updates[client_id])
            elif attack_type == 'data_poisoning':
                poisoned_updates[client_id] = self.data_poisoning_attack(local_updates[client_id])
            elif attack_type == 'gradient_ascent':
                poisoned_updates[client_id] = -2 * local_updates[client_id]
            elif attack_type == 'backdoor':
                poisoned_updates[client_id] = self.backdoor_attack(local_updates[client_id])

        return poisoned_updates, byzantine_clients

    def model_poisoning_attack(self, genuine_update):
        noise_scale = np.linalg.norm(genuine_update) * 2
        malicious_direction = np.random.randn(*genuine_update.shape)
        malicious_direction = malicious_direction / np.linalg.norm(malicious_direction)
        return genuine_update + noise_scale * malicious_direction

    def data_poisoning_attack(self, genuine_update):
        flip_probability = 0.4
        poisoned_update = genuine_update.copy()
        flip_mask = np.random.random(genuine_update.shape) < flip_probability
        poisoned_update[flip_mask] *= -1.5
        return poisoned_update

    def backdoor_attack(self, genuine_update):
        backdoor_pattern = np.ones_like(genuine_update) * 0.1
        backdoor_strength = 3.0
        return genuine_update + backdoor_strength * backdoor_pattern

    def detect_byzantine_clients(self, local_updates):
        update_matrix = np.array(local_updates)
        mean_update = np.mean(update_matrix, axis=0)

        anomaly_scores = []
        for i, update in enumerate(local_updates):
            cosine_dist = cosine(update, mean_update)
            magnitude_ratio = np.linalg.norm(update) / (np.linalg.norm(mean_update) + 1e-10)
            gradient_entropy = entropy(np.abs(update) + 1e-10)

            statistical_distance = np.linalg.norm(update - mean_update)
            anomaly_score = cosine_dist + abs(np.log(magnitude_ratio)) + gradient_entropy/10 + statistical_distance/100
            anomaly_scores.append(anomaly_score)

        threshold = np.percentile(anomaly_scores, (1 - self.byzantine_ratio) * 100)
        detected_byzantines = [i for i, score in enumerate(anomaly_scores) if score > threshold]

        return detected_byzantines, anomaly_scores

    def byzantine_robust_aggregation(self, local_updates, method='trimmed_mean'):
        update_matrix = np.array(local_updates)

        if method == 'trimmed_mean':
            trim_ratio = self.byzantine_ratio + 0.1
            trimmed_updates = []
            for j in range(update_matrix.shape[1]):
                column = update_matrix[:, j]
                sorted_col = np.sort(column)
                trim_count = int(len(sorted_col) * trim_ratio)
                trimmed_col = sorted_col[trim_count:-trim_count] if trim_count > 0 else sorted_col
                trimmed_updates.append(np.mean(trimmed_col))
            return np.array(trimmed_updates)

        elif method == 'coordinate_median':
            return np.median(update_matrix, axis=0)

        elif method == 'krum':
            distances = np.zeros((self.num_clients, self.num_clients))
            for i in range(self.num_clients):
                for j in range(i+1, self.num_clients):
                    dist = np.linalg.norm(local_updates[i] - local_updates[j])
                    distances[i, j] = distances[j, i] = dist

            krum_scores = []
            for i in range(self.num_clients):
                closest_distances = np.sort(distances[i])[1:self.num_clients - int(self.byzantine_ratio * self.num_clients)]
                krum_scores.append(np.sum(closest_distances))

            selected_client = np.argmin(krum_scores)
            return local_updates[selected_client]

        else:
            return np.mean(update_matrix, axis=0)

    def adaptive_trust_scoring(self, client_id, prediction_accuracy, consistency_score):
        learning_rate = 0.1
        accuracy_weight = 0.7
        consistency_weight = 0.3

        performance_score = accuracy_weight * prediction_accuracy + consistency_weight * consistency_score
        self.trust_scores[client_id] = (1 - learning_rate) * self.trust_scores[client_id] + learning_rate * performance_score
        self.trust_scores[client_id] = np.clip(self.trust_scores[client_id], 0.1, 1.0)

    def differential_privacy_mechanism(self, local_update, epsilon=1.0, delta=1e-5):
        sensitivity = np.linalg.norm(local_update, ord=2)
        noise_scale = sensitivity * np.sqrt(2 * np.log(1.25/delta)) / epsilon
        gaussian_noise = np.random.normal(0, noise_scale, local_update.shape)
        return local_update + gaussian_noise

    def secure_federated_training(self, client_data, attack_type='model_poisoning', defense_method='trimmed_mean'):
        local_updates = []
        for i, (X_client, y_client) in enumerate(client_data):
            if len(X_client) > 0:
                X_scaled = self.scaler.fit_transform(X_client)
                self.client_models[i].fit(X_scaled, y_client)
                local_update = self.client_models[i].coef_.flatten()
                local_update = self.differential_privacy_mechanism(local_update)
                local_updates.append(local_update)
            else:
                local_updates.append(np.zeros(X_client.shape[1] if len(client_data) > 0 else 10))

        poisoned_updates, true_byzantines = self.simulate_byzantine_attacks(local_updates, attack_type)
        detected_byzantines, anomaly_scores = self.detect_byzantine_clients(poisoned_updates)

        robust_aggregate = self.byzantine_robust_aggregation(poisoned_updates, defense_method)
        self.aggregation_history.append(robust_aggregate)

        detection_accuracy = len(set(detected_byzantines) & set(true_byzantines)) / max(len(true_byzantines), 1)
        false_positive_rate = len(set(detected_byzantines) - set(true_byzantines)) / max(self.num_clients - len(true_byzantines), 1)

        return {'aggregate': robust_aggregate, 'detection_accuracy': detection_accuracy,
                'false_positive_rate': false_positive_rate, 'anomaly_scores': anomaly_scores,
                'trust_scores': self.trust_scores.copy()}